# Cleaning The Ontario Sunshine List Data

<a id="table-of-contents"></a>

This notebook was used to clean Ontario's public sector salary disclosure data (also known as the Ontario sunshine list). The cleaned dataset can be found [here](https://www.kaggle.com/sahidvelji/the-ontario-sunshine-list), and an EDA of the 2019 data can be found [here](https://www.kaggle.com/sahidvelji/the-ontario-sunshine-list-2019-eda).

## Table of contents

<p style="line-height: 1.6em;">
    <a href="#loading-data">1. Loading the data</a><br>
    <a href="#renaming-columns">2. Renaming columns</a><br>
    <a href="#data-types">3. Data types</a><br>
    <a href="#missing-values">4. Missing values</a><br>
    <a href="#concat-data">5. Concatenating the dataframes</a><br>
    <a href="#write-to-csv">6. Writing to CSV</a><br>
</p>

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import shutil
from IPython.display import display, HTML
import plotly.express as px
from pathlib import Path

YEARS = range(1996, 2020)

px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = ['steelblue']
MODE_BAR_BUTTONS = ['toImage', 'zoom2d', 'pan2d', 'select2d', 'lasso2d',
                    'zoomIn2d', 'zoomOut2d', 'autoScale2d', 'resetScale2d',
                    'toggleSpikelines', 'hoverClosestCartesian', 'hoverCompareCartesian']
CONFIG = {
    'modeBarButtonsToRemove': ['pan2d', 'select2d', 'lasso2d', 'toggleSpikelines']
}

# Loading the data

In [ ]:
path = Path('/kaggle/input/the-ontario-sunshine-list-raw-data')  
filenames = os.listdir(path)
filenames

While downloading the data from the government of Ontario website, I realized that the filenames were inconsistent. Even worse, there are two files that appear to both be data from 2018: `en-2018-pss-compendium.csv` and `en-2018-pss-compendium-20191223.csv`. Let's examine these two files.

In [ ]:
pd.read_csv(path/'en-2018-pssd-compendium.csv', nrows=5)

In [ ]:
pd.read_csv(path/'en-2018-pssd-compendium-20191223.csv', nrows=5)

It turns out that `en-2018-pss-compendium.csv` is actually data from 2017 and `en-2018-pss-compendium-20191223.csv` is data from 2018. The input directory is read-only data, so we cannot rename files here. However, we will use a dictionary to organize the files by year after loading them into dataframes. First, we'll create a regular expression to extract the year from each filename. 

In [ ]:
p = re.compile('(?:^|-)(\d{4})\D')

That should do it. The year appears either at the beginning of the filename or after a dash. The character following the year is either a dash, a dot, or an underscore. This means we have a non-numeric character following the year. The regular expression was constructed based on these observations: first we match the beginning of a string or a dash in a non-capturing group. Then, we match 4 digits in a capturing group. This is followed by a single non-numeric digit.

Unfortunately, we encounter a `UnicodeDecodeError` if we try to load the data with the standard utf-8 encoding for some of the files. Instead, we'll use latin1 encoding if the default fails. The error occurs due to characters such as "é" in the data.

In [ ]:
def read_csv(filename):
    try:
        return pd.read_csv(path/filename, encoding='utf-8')
    except UnicodeDecodeError:
        return pd.read_csv(path/filename, encoding='latin1')

In [ ]:
pss = {}
for filename in filenames:
    if filename == 'en-2018-pssd-compendium.csv':
        pss[2017] = read_csv(filename)
        continue
    m = p.search(filename)
    year = int(m.group(1))
    pss[year] = read_csv(filename)

Now we have a dictionary where each key is a calendar year and the value is the corresponding dataframe. The `describe` method is a useful way of displaying a summary table of the data. Examining all 24 tables one by one wouldn't be very efficient but we'll display the tables here for reference purposes anyways.

In [ ]:
for year in YEARS:
    display(HTML(pss[year].describe(include='all').to_html()))

The Salary Paid and Taxable Benefits columns should have type `float` but it looks like they are currently strings because of the dollar signs and commas. We will fix this later.

First, we will make sure that the column names of all dataframes match.

Second, we will deal with the data types of the columns. As mentioned above, we expect the Salary Paid and Taxable Benefits columns to have type `float`. The Calendar Year column should have type `int` and the rest of the columns should have type `string`.

Then, we will check for missing values and see if there is a good way of imputing them. Finally, we will explore the possibility of concatenating the dataframes. After all, they should have the same number of columns, the same column names and data types, and one dataframe is easier to work with than 24 separate ones.

<a id="renaming-columns"></a>
[Return to table of contents](#table-of-contents)

# Renaming columns

We will check to make sure that every dataframe has the same column names using the 2019 columns as a reference.

In [ ]:
refcols = pss[2019].columns
for year in YEARS:
    if not refcols.equals(pss[year].columns):
        print(year, pss[year].columns.tolist(), sep='\n', end='\n\n')

print("2019", refcols.tolist(), sep='\n')

There are numerous inconsistencies:

- There is an extra column in the 1996 dataframe. 
- The 2001 dataframe has Surname instead of Last Name and Position instead of Job Title. 
- There are trailing whitespaces in the Salary Paid column in the 2009, 2010, and 2011 dataframes.
- In the 2014 dataframe, the second words of "Last name", "Job title", and "Calendar year" are not capitalized. 

We will first examine the extra column in the 1996 dataframe and drop it if appropriate.

In [ ]:
pss[1996].head()

In [ ]:
pss[1996]['Unnamed: 8'].isna().all()

This unnamed column is filled with `NaN` values, meaning that we can safely drop this column.

In [ ]:
pss[1996] = pss[1996].drop(columns='Unnamed: 8')

Now that every dataframe has the same number of columns, let's rename all columns to match the columns of the 2019 dataframe.

In [ ]:
for year in YEARS:
    pss[year].columns = refcols

The Calendar Year column may seem redundant because of the file name. For example, we know that all of the data in `tbs-pssd-compendium-en-utf8-2019.csv` is for the calendar year 2019. However, we will not drop this column yet in case we later want to concatenate the dataframes.

<a id="data-types"></a>
[Return to table of contents](#table-of-contents)

# Data types

Here, we will examine the data types of the columns and ensure that they match across all dataframes.

In [ ]:
for year in YEARS:
    print(year, pss[year].dtypes, sep='\n', end='\n\n')

Unfortunately, we discover more inconsistencies. Quickly scrolling through the output tells us that for most dataframes, the Calendar Year column has data type `int`, except for the 2016 dataframe. Also, the dataframes for 2012 and 2013 have type `float` for the Salary Paid and Taxable Benefits columns while the other dataframes have type `object` for these columns.

Let's find out why the Calendar Year column in the 2016 dataframe doesn't have type `int`.

In [ ]:
pss[2016]['Calendar Year'].nunique()

That doesn't look promising. We expect a single unique value for the Calendar Year column, but we have 38 unique values instead.

In [ ]:
wrong_year = pss[2016][pss[2016]['Calendar Year'] != '2016']
wrong_year.shape[0]

In [ ]:
wrong_year.head(20)

Unfortunately, 65 rows are affected. We need all values in the Calendar Year column to be 2016, meaning that we need to get rid of the current values somehow. It seems as though they are job titles. For example, the third row has value "Deputy Minister" in the Calendar Year column. The corresponding job title is "Housing", which surely cannot be a job title. On the other hand, we see that the row below it has job title "Director". But, the value in the Calendar Year column is "seconded to Stevenson Memorial Hospital as CEO".

From the [2015 salary disclosure page](https://www.ontario.ca/page/public-sector-salary-disclosure-2015-all-sectors-and-seconded-employees):
>Someone who is “seconded” has a job in a public sector organization (other than an Ontario government ministry), but currently works within a government ministry. The organization pays the person’s salary and benefits and the ministry reimburses the organization.

Since the dataframes at least have the same column names at this point, we will concatenate them to make our job easier for this section.

In [ ]:
pss_comb = pd.concat([pss[year] for year in YEARS]).copy()

Let's take a closer look at a few people's job titles and corresponding calendar year values in attempt to find a solution to this problem.

In [ ]:
pss_comb[pss_comb['Last Name'].str.contains('Fagan', case=False) & pss_comb['First Name'].str.contains('Thomas', case=False)]

Fagan's job title in 2016 is very likely "Member", just as in subsequent years.

In [ ]:
pss_comb[pss_comb['Last Name'].str.contains('leblanc', case=False) & pss_comb['First Name'].str.contains('Laurie', case=False)]

It is clear that in 2016, Laurie Leblanc's job title should be "Deputy Minister". So, we could replace "Housing" with "Deputy Minister". But, the problem is that this is not a general solution. For example, is Dora Cavallo-Medved's (in the wrong_year dataframe, third row from the bottom) job title "Sessional Lecturer I" or "Course Developer"?

In [ ]:
pss_comb[pss_comb['Last Name'].str.contains('levac', case=False) & pss_comb['First Name'].str.contains('jody', case=False)]

In 2011, 2013, and 2018, the job titles are of the form "English / French", unlike the other years. Even worse, in 2011, both the employer and job title are of the form "English / French" but in 2013, only the job title is. Even the name columns are inconsistent: from 2007 to 2013, the first and last names are in upper case, unlike 2014 to 2019. From 2014 to 2017, the first name column has an initial "J.", which is not consistent with the rest of the years.

This one gives us an idea. We could append the string in the Calendar Year column to the job title column, separated by a semicolon, as in 2015.

In [ ]:
pss[2016].loc[pss[2016]['Calendar Year'] != '2016', 'Job Title'] = wrong_year['Job Title'].str.cat(wrong_year['Calendar Year'], sep='; ')
pss[2016].loc[pss[2016]['Calendar Year'] != '2016', 'Calendar Year'] = '2016'
pss[2016]['Calendar Year'] = pss[2016]['Calendar Year'].astype('int')
pss[2016].dtypes

Had we dropped the Calendar Year column before examining it closely, we would have lost job title information for 65 rows in the 2016 dataframe. Therefore, it is always a good idea to examine a column before dropping it when cleaning data.

Before moving on, we will take a look at the Calendar Year values in each dataframe to make sure that there are no surprises.

In [ ]:
for year in YEARS:
    if pss[year]['Calendar Year'].nunique() > 1:
        print(year, pss[year]['Calendar Year'].unique(), sep='\n')

It looks like the Calendar Year column in the 2015 dataframe has values other than 2015. Another important lesson: if we had assumed that there were no mistakes in the Calendar Year column, we could make some serious errors. For example, if we decided to concatenate the dataframes and perform a groupby operation on the Calendar Year column, then several rows would be incorrectly grouped into 2016, 2017, and 2018.

In [ ]:
wrong_year_2015 = pss[2015][pss[2015]['Calendar Year'] != 2015]
wrong_year_2015

Only a few of the rows are affected. Since this data was released in 2016 for the calendar year 2015, the above values don't make sense. We will correct these now.

In [ ]:
pss[2015].loc[wrong_year_2015.index, 'Calendar Year'] = 2015

Now that the Calendar Year column has the same data type across all dataframes, we will move on to examine the Salary Paid and Taxable Benefits columns. We already saw that we need to remove dollar signs and commas. But, is that everything? We will check for other non-numeric characters. In the two code cells below, we search for any non-numeric characters excluding dollar signs, commas, periods, and spaces.

In [ ]:
pss_comb = pd.concat([pss[year] for year in YEARS]).copy().reset_index(drop=True)

salary_nonnum = pss_comb['Salary Paid'].str.extractall('([^$\d.,\s])').drop_duplicates()
salary_nonnum

In [ ]:
tax_ben_nonnum = pss_comb['Taxable Benefits'].str.extractall('([^$\d.,\s])').drop_duplicates()
tax_ben_nonnum

It looks like we have dashes in the Taxable Benefits column. We will need to remove these before converting to this column to the `float` data type. Before we do so, we should take a look at how the dash appears in the data.

In [ ]:
idx = tax_ben_nonnum.reset_index(level='match', drop=True).index
pss_comb.loc[idx, 'Taxable Benefits'].unique()

The dash only appears in the form "$-".

Once we have replaced all dollar signs and commas, we will replace all dashes with `np.nan` since these are missing values.

In [ ]:
for year in YEARS:
    pss[year]['Salary Paid'] = (pss[year]['Salary Paid']
                                .replace('[$,]', '', regex=True)
                                .replace('-', np.nan)
                                .astype('float')
                               )
    pss[year]['Taxable Benefits'] = (pss[year]['Taxable Benefits']
                                     .replace('[$,]', '', regex=True)
                                     .replace('-', np.nan)
                                     .astype('float')
                                    )

Finally, we convert the columns that have data type `object` to `string`. We may do so by calling `convert_dtypes()` on each dataframe. This will convert the columns to the best possible data types. The [Pandas documentation](https://pandas.pydata.org/pandas-docs/stable/getting_started/basics.html#dtypes) explains why this is a good idea:

> Pandas has two ways to store strings.
> 1. object dtype, which can hold any Python object, including strings.
> 1. StringDtype, which is dedicated to strings.
> 
> Generally, we recommend using StringDtype.
> Finally, arbitrary objects may be stored using the object dtype, but should be avoided to the extent possible (for performance and interoperability with other libraries and methods).

In [ ]:
for year in YEARS:
    pss[year] = pss[year].convert_dtypes()
    print(year, pss[year].dtypes, sep='\n', end='\n\n')

Each column now has the desired data type. Since we are writing this data to a CSV, we will need to convert the data types again when we explore this data in a different notebook. So, why convert data types here? We ran into several issues while attempting to do so. The idea is to take care of those issues here instead of in an analysis notebook.

<a id="missing-values"></a>
[Return to table of contents](#table-of-contents)

# Missing values

In [ ]:
for year in YEARS:
    if pss[year].isna().sum().sum() != 0:
        print(year, pss[year].isna().sum(), sep='\n', end='\n\n')

Apart from the 6980 in the 2015 dataframe, there are not many missing values. Let's first find out why there are so many missing values in the Taxable Benefits column in 2015. Maybe those missing values are supposed to be zeros.

In [ ]:
pss[2015]['Taxable Benefits'].eq(0).sum()

It seems as if every employee received some amount of taxable benefits in 2015. This seems unlikely, but we could take a look at the data from other years and compare.

In [ ]:
pss_comb = (pd.concat([pss[year] for year in YEARS])
            .copy()
            .reset_index(drop=True)
           )
no_tax_ben = (pss_comb
              .loc[pss_comb['Taxable Benefits'].eq(0), 'Calendar Year']
              .value_counts()
              .reindex(list(YEARS), fill_value=0)
              .to_frame()
              .reset_index()
              .rename(columns={'index': 'Calendar Year', 'Calendar Year': 'Number of employees'})
             )

In [ ]:
fig = px.scatter(no_tax_ben, x='Calendar Year', y='Number of employees')
fig.update_traces(mode='lines+markers',
                  hovertemplate=
                  '<b>%{x}</b><br>'+
                  'Number of employees: <b>%{y}</b>'
                 )
fig.update_layout(title='Number of employees that did not receive taxable benefits by calendar year',
                  xaxis_title='Calendar Year',
                  yaxis_title="Number of employees",
                  yaxis_tickformat=',',
                  hoverlabel_bgcolor="white",
                  hoverlabel_font_size=14,
                  hovermode="x",
                  yaxis_zerolinecolor='grey',
                  yaxis_zerolinewidth=1
                 )
fig.show(config=CONFIG)

Based on the above plot, we can reasonably assume that the missing values for the 2015 Taxable Benefits column must be zeros. We will fill in the missing values and then plot the data again.

In [ ]:
pss[2015]['Taxable Benefits'].fillna(0.0, inplace=True)
no_tax_ben.loc[no_tax_ben['Calendar Year'].eq(2015), 'Number of employees'] = pss[2015]['Taxable Benefits'].eq(0).sum()

In [ ]:
fig = px.scatter(no_tax_ben, x='Calendar Year', y='Number of employees')
fig.update_traces(mode='lines+markers',
                  hovertemplate=
                  '<b>%{x}</b><br>'+
                  'Number of employees: <b>%{y}</b>'
                 )
fig.update_layout(title='Number of employees that did not receive taxable benefits by calendar year',
                  xaxis_title='Calendar Year',
                  yaxis_title="Number of employees",
                  yaxis_tickformat=',',
                  hoverlabel_bgcolor="white",
                  hoverlabel_font_size=14,
                  hovermode="x",
                  yaxis_zerolinecolor='grey',
                  yaxis_zerolinewidth=1
                 )
fig.show(config=CONFIG)

The plot makes more sense now. The number of employees that did not receive taxable benefits in 2015 is between the corresponding values for 2014 and 2016.

The 2016 data has one missing value in the Taxable Benefits column.

In [ ]:
null_2016 = pss[2016][pss[2016]['Taxable Benefits'].isna()]
null_2016

We will look for "James Malenfant" in other years in hopes of being able to impute this value.

In [ ]:
pss_comb[pss_comb['Last Name'].eq('Malenfant') & pss_comb['First Name'].eq('James')]

Since James Malenfant does not usually receive taxable benefits, it is reasonable to assume that he didn't receive taxable benefits in 2016 either.

In [ ]:
pss[2016].loc[null_2016.index, 'Taxable Benefits'] = 0.0

The 2013 data has one missing value in the First Name column.

In [ ]:
null_2013 = pss[2013][pss[2013]['First Name'].isna()]
null_2013

In [ ]:
pss_comb[pss_comb['Last Name'].str.contains('^li$', case=False) & pss_comb['Employer'].str.contains('eHealth') & pss_comb['Job Title'].str.contains('privacy', case=False)]

We can safely assume that Li's first name is Na based on the record from 2014. It looks like the first name "NA" was marked as a missing value in the 2013 data. The first name "Na" in the 2014 data was not marked as a missing value. From the `pandas.read_csv` [documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html):

> **na_values: scalar, str, list-like, or dict, optional**   
> Additional strings to recognize as NA/NaN. If dict passed, specific per-column NA values. By default the following values are interpreted as NaN: ‘’, ‘#N/A’, ‘#N/A N/A’, ‘#NA’, ‘-1.#IND’, ‘-1.#QNAN’, ‘-NaN’, ‘-nan’, ‘1.#IND’, ‘1.#QNAN’, ‘<NA>’, ‘N/A’, ‘NA’, ‘NULL’, ‘NaN’, ‘n/a’, ‘nan’, ‘null’.
    
This means that LI's first name "NA" in the 2013 data was likely never really a missing value to begin with. Let's find out what the original first name value was in 2013. We can do so by passing the argument `False` to the `na_filter` parameter when calling `pd.read_csv`.

In [ ]:
pss2013 = pd.read_csv(path/'pssd-en-2013.csv', na_filter=False)
pss2013[pss2013['Last Name'].str.contains('^li$', case=False) & pss2013['Employer'].str.contains('eHealth') & pss2013['Job Title'].str.contains('privacy', case=False)]

Indeed we see that the first name was originally "NA", but was marked as a missing value. We will set the missing first name to "NA" instead of "Na" in order to be consistent, since names in the 2013 data seem to be in upper case only.

In [ ]:
pss[2013].loc[null_2013.index, 'First Name'] = 'NA'

The 1998 data has one missing value in the first name column.

In [ ]:
null_1998 = pss[1998][pss[1998]['First Name'].isna()]
null_1998

In [ ]:
pss_comb[pss_comb['Last Name'].str.contains('donnelly', case=False) & pss_comb['Employer'].str.contains('hydro', case=False)]

Based on the record from 1999, we can safely assume that Donnelly's first initial is "N". However, just as we saw above, it may be the case that `pd.read_csv` marked Donnelly's first name as a missing value.

In [ ]:
pss1998 = pd.read_csv(path/'en-1998-pssd.csv', encoding='latin1', na_filter=False)
pss1998[pss1998['Last Name'].str.contains('donnelly', case=False) & pss1998['Employer'].str.contains('hydro', case=False)]

Just like before, a first name of "NA" was marked as a missing value. Donnelly's first name is probably not "NA" though. It's more likely that these are initials. It's not uncommon in the 1998 data to have 2 letter initials in the First Name column.

In [ ]:
pss[1998][pss[1998]['First Name'].str.len().eq(2)].head()

Therefore, we'll set the missing first name to "NA", as it was originally.

In [ ]:
pss[1998].loc[null_1998.index, 'First Name'] = 'NA'

The 1997 data has one missing value in the Job Title column.

In [ ]:
null_1997 = pss[1997][pss[1997]['Job Title'].isna()]
null_1997

In [ ]:
pss_comb[pss_comb['Last Name'].str.contains('walker', case=False) & pss_comb['Employer'].str.contains('ontario hydro', case=False)]

Since they have the same job title, "Walker, D G" from 1997 and "Walker, G" from 1998 seem to be the same person. The other "Walker, G" has the job title "Maintenance Superintendent" in 1998, which is one year after "Walker, G J" in 1997. My best guess then, is to say that "Walker, G J" has job title "Maintenance Superintendent". This [external page](https://www.ontariosunshinelist.com/people/fqpyng) seems to support my claim.

In [ ]:
pss[1997].loc[null_1997.index, 'Job Title'] = 'Maintenance Superintendent'

The 1996 data has one missing value in the First Name column.

In [ ]:
null_1996 = pss[1996][pss[1996]['First Name'].isna()]
null_1996

In [ ]:
pss_comb[pss_comb['Last Name'].str.contains('yearwood', case=False) & pss_comb['Calendar Year'].le(2010)]

This is not helpful. Let's find out what the original value of the first name was.

In [ ]:
pss1996 = pd.read_csv(path/'en-1996-pssd.csv', encoding='latin1', na_filter=False)
pss1996[pss1996['Last Name'].str.contains('yearwood', case=False)]

Again, a first name of "NA" was interpreted to be a missing value by the `pd.read_csv` function. This is the last missing value. If we had many missing values, we would have to deal with them in a less cumbersome way. For example, we could specify that we only want empty strings to be interpreted as missing values when reading in the data with `pd.read_csv`.

Just like before, we'll keep the original first name of "NA", since it's not uncommon to have 2 letter initials in the 1996 data.

In [ ]:
pss[1996][pss[1996]['First Name'].str.len().eq(2)].head()

In [ ]:
pss[1996].loc[null_1996.index, 'First Name'] = 'NA'

<a id="concat-data"></a>
[Return to table of contents](#table-of-contents)

# Concatenating the dataframes

In [ ]:
pss_comb = pd.concat([pss[year] for year in YEARS]).copy()

The data spans from 1996 to 2019 and as we saw throughout this notebook, there were many inconsistencies. Here is just one of many examples:

In [ ]:
pss_comb[pss_comb['Last Name'].str.contains('malenfant', case=False) & pss_comb['First Name'].str.contains('andrew', case=False)]

I think this is a good example because it displays several inconsistencies. It is clear that these records all belong to the same person. In the Sector column, some rows use the ampersand (&) instead of the word "and". In the Last Name and First Name columns, some values are in upper case while others are not. Also, in 2019, the first name is just "Andrew" instead of "Andrew Derek".

It would be nice if each person were given some sort of unique id. This way, even is a person's name changes, or if we see something like the example above, we would be able to easily tell whether two people on the list are the same person.

It is much easier to work with one dataframe as opposed to 24 and the calendar year column can always be used to separate the data again if needed. Another reason one CSV file is preferred over 24 is that file descriptions and column descriptions in the resulting dataset will only need to be specified one time instead of 24 times. I will also add a note in the dataset description about the inconsistency of the data across calendar years.

<a id="write-to-csv"></a>
[Return to table of contents](#table-of-contents)

# Write to CSV

In [ ]:
pss_comb

We sort the data as a final step before writing the data to a CSV file.

In [ ]:
pss_comb.sort_values(['Calendar Year', 'Sector', 'Employer', 'Last Name', 'First Name']).to_csv('pssd.csv', index=False)